# Skintelligent Recommendation System

In [1]:
# imports
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# loading dataset
full_data = pd.read_csv('../data/reviews_all_products.csv')
full_data.head()

/var/folders/_s/x14mjmjn7qj5qh8bhxln436r0000gn/T/ipykernel_11227/3277208589.py:2: DtypeWarning: Columns (1,28) have mixed types. Specify dtype option on import or set low_memory=False.
  full_data = pd.read_csv('../data/reviews_all_products.csv')


,Unnamed: 0,author_id,review_rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,...,1,0,0,['Clean at Sephora'],Skincare,Cleansers,NaN,0,NaN,NaN
1,1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
2,2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
3,3,6083038851,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
4,4,47056667835,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0


In [3]:
full_data.shape

(975094, 45)

In [4]:
print(full_data.columns.tolist())

['Unnamed: 0', 'author_id', 'review_rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'review_price', 'product_name_y', 'brand_id', 'brand_name_y', 'loves_count', 'product_rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'product_price', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price']


### Feature Engineering
---

In [ ]:
# helpfullness weighted rating
full_data['helpfulness'].fillna(0, inplace=True)

full_data['weighted_rating'] = (
    full_data['review_rating'] *
    (1 + full_data['total_pos_feedback_count']) /
    (1 + full_data['total_feedback_count'])
)

In [6]:
full_data[['weighted_rating', 'helpfulness', 'review_rating', 'total_pos_feedback_count']].head()

,weighted_rating,helpfulness,review_rating,total_pos_feedback_count
0,15,1.0,5,2
1,1,0.0,1,0
2,5,0.0,5,0
3,5,0.0,5,0
4,5,0.0,5,0


**Colummn to find how popular this product is among people with your skin type**

In [7]:
# group by product_id and skin_type to get count of each skin type for each product
skin_match = full_data.groupby(['product_id', 'skin_type']).size().unstack(fill_value=0)

# convert counts to ratios
skin_match_ratio = skin_match.div(skin_match.sum(axis=1), axis=0)

### Product Level Aggregation
---

In [8]:
product_stats = full_data.groupby('product_id').agg({
    'product_name': 'first',
    'brand_name': 'first',
    'product_rating': 'mean',
    'review_rating': ['mean', 'std', 'count'],
    'weighted_rating': 'mean',
    'loves_count': 'mean',
    'reviews': 'mean',
    'ingredients': 'first',
    'primary_category': 'first',
    'secondary_category': 'first',
    'tertiary_category': 'first',
    'highlights': 'first',
    'product_price': 'mean',
    'sephora_exclusive': 'mean',
    'new': 'mean'
}).reset_index()

product_stats.columns = [
    'product_id','product_name','brand_name','product_rating',
    'avg_review_rating','rating_std','review_count',
    'weighted_rating','loves_count','review_total',
    'ingredients','primary_category','secondary_category',
    'tertiary_category','highlights','price','exclusive','is_new'
]

### NLP Pipeline
---

In [9]:
nlp_df = full_data[['product_id', 'review_text', 'review_rating', 'skin_type', 'ingredients']].copy()

nlp_df = nlp_df.dropna(subset=['product_id', 'review_text'])

In [10]:
# NLP pipeline
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

nlp_df['clean_review'] = nlp_df['review_text'].apply(clean_text)

In [19]:
!pip install vaderSentiment

In [20]:
# sentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

In [22]:
def get_sentiment(text):
    return sia.polarity_scores(text)['compound']

nlp_df['sentiment_score'] = nlp_df['clean_review'].apply(get_sentiment)

In [23]:
# sentiment labeling
def label_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

nlp_df['sentiment_label'] = nlp_df['sentiment_score'].apply(label_sentiment)

In [24]:
# aggregate to product level
avg_sentiment = nlp_df.groupby('product_id')['sentiment_score'].mean().reset_index()
avg_sentiment.rename(columns={'sentiment_score': 'avg_sentiment'}, inplace=True)

polarization = nlp_df.groupby('product_id')['sentiment_score'].std().reset_index()
polarization.rename(columns={'sentiment_score': 'polarization_score'}, inplace=True)

polarization['polarization_score'] = polarization['polarization_score'].fillna(0)

product_sentiment = avg_sentiment.merge(polarization, on='product_id', how='left')

In [25]:
# summary + ingredients
from collections import Counter

def make_summary(group):
    pos = group[group['sentiment_label'] == 'positive']['review_text']
    neg = group[group['sentiment_label'] == 'negative']['review_text']

    summary_parts = []

    if len(pos) > 0:
        summary_parts.append("Most positive: " + str(pos.iloc[0])[:120])
    if len(neg) > 0:
        summary_parts.append("Negative: " + str(neg.iloc[0])[:120])

    return " | ".join(summary_parts) if summary_parts else "No strong signals."

summary_df = nlp_df.groupby('product_id').apply(make_summary).reset_index()
summary_df.columns = ['product_id', 'summary_text']

/var/folders/_s/x14mjmjn7qj5qh8bhxln436r0000gn/T/ipykernel_11227/335351767.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_df = nlp_df.groupby('product_id').apply(make_summary).reset_index()


In [26]:
# merging to product_stats
product_stats = product_stats.merge(product_sentiment, on='product_id', how='left')

# optional extras
product_stats = product_stats.merge(summary_df, on='product_id', how='left')

### Core Scoring Features
---

#### Popularity

In [27]:
product_stats['popularity'] = np.log1p(
    product_stats['loves_count'] + product_stats['review_count']
)

product_stats['popularity_score'] = (
    (product_stats['popularity'] - product_stats['popularity'].min()) /
    (product_stats['popularity'].max() - product_stats['popularity'].min())
)

#### Rating (Confidence Weighted)

In [28]:
product_stats['confidence'] = (
    np.log1p(product_stats['review_count']) /
    np.log1p(product_stats['review_count'].max())
)

product_stats['rating_score'] = (product_stats['avg_review_rating'] - 1) / 4

product_stats['adjusted_rating'] = (
    product_stats['rating_score'] * product_stats['confidence']
)

#### "Worth The Hype" Score

In [29]:
product_stats['rating_std'] = product_stats['rating_std'].fillna(0)
product_stats['hype_score'] = 1 / (1 + product_stats['rating_std'])

#### Highlight Score

In [ ]:
import ast

def highlight_score(highlights_str, user_profile):
    """
    Compute a score based on how well product highlights match user's skin concerns.
    """
    if pd.isna(highlights_str) or not highlights_str:
        return 0.5  # neutral
    
    # Normalize and convert string to list
    try:
        highlights = ast.literal_eval(highlights_str)
    except:
        highlights = [highlights_str]
    
    highlights = [h.lower() for h in highlights]
    
    score = 0
    concerns = user_profile.get('skin_concerns', [])
    
    for concern in concerns:
        concern = concern.lower()
        # simple matching
        if any(concern in h for h in highlights):
            score += 1
    
    # normalize to 0-1
    if concerns:
        return score / len(concerns)
    else:
        return 0.5

### Ingredient Model
---

In [31]:
ingredient_risk = pd.read_csv('../data/ingredient_risk_table.csv')

In [32]:
# consistent comparisons
def normalize_string(s):
    if pd.isna(s):
        return ""
    return str(s).strip().lower().replace(" ", "").replace("-", "")

# ingredient compatibility function
def ingredient_compatibility_score(ingredient_string, user_profile, ingredient_risk):
    """
    Compute a compatibility score (0-1) based on ingredient risks and user profile.
    """
    if pd.isna(ingredient_string) or not ingredient_string:
        return 0.5  # neutral score if no ingredients listed

    # Normalize ingredients
    ingredients = [ing.strip().lower() for ing in ingredient_string.split(",")]

    score = 1.0
    penalties = 0

    # Risk weights
    risk_weight = {
        "acne_risk": 0.2,
        "irritation_risk": 0.25,
        "dry_skin_risk": 0.15,
        "sensitivity_risk": 0.25
    }

    # Apply risk-based penalties
    for _, row in ingredient_risk.iterrows():
        ing_name = row["ingredient_name"].lower().strip()
        if ing_name in ingredients:

            # Acne-prone
            if user_profile.get("acne_prone", False):
                penalties += row["acne_risk"] * risk_weight["acne_risk"]

            # Sensitive skin
            if user_profile.get("skin_sensitivity", "Low") in ["High", "Very High"]:
                penalties += row["irritation_risk"] * risk_weight["irritation_risk"]

            # Dry skin
            if user_profile.get("skin_type", "").lower() == "dry":
                penalties += row["dry_skin_risk"] * risk_weight["dry_skin_risk"]

    # Fragrance allergy
    if user_profile.get("fragrance_allergy", False) and "fragrance" in ingredients:
        penalties += 0.3

    final_score = score - penalties
    return max(min(final_score, 1), 0)

In [33]:
# ingredient reason codes function
def ingredient_reason_codes(ingredient_string, user_profile, ingredient_risk):
    """
    Generate reason codes explaining ingredient risks for the user.
    """
    reasons = []
    if pd.isna(ingredient_string) or not ingredient_string:
        return reasons

    ingredients = [ing.strip().lower() for ing in ingredient_string.split(",")]

    for _, row in ingredient_risk.iterrows():
        ing_name = row["ingredient_name"].lower().strip()
        if ing_name in ingredients:

            # Acne-prone
            if user_profile.get("acne_prone", False) and row["acne_risk"] > 0:
                reasons.append(f"Ingredient {row['ingredient_name']} may trigger acne")

            # Sensitive skin
            if user_profile.get("skin_sensitivity", "Low") in ["High", "Very High"] and row["irritation_risk"] > 0:
                reasons.append(f"Ingredient {row['ingredient_name']} may irritate sensitive skin")

            # Dry skin
            if user_profile.get("skin_type", "").lower() == "dry" and row["dry_skin_risk"] > 0:
                reasons.append(f"Ingredient {row['ingredient_name']} may worsen dryness")

    # Fragrance allergy
    if user_profile.get("fragrance_allergy", False) and "fragrance" in ingredients:
        reasons.append("Contains fragrance (allergy warning)")

    return reasons

### Collaborative Filtering (Similar Users Algorithm)
---

1. Profile Similarity – Encodes skin type, tone, sensitivity, eye/hair color into one-hot vectors and computes cosine similarity.
2. Rating Similarity – Cosine similarity based on weighted_rating history.
3. Hybrid Similarity – Weighted combination (0.6 profile + 0.4 rating by default, adjustable).
4. Top N Users – Only considers top similar users to avoid noise.
5. Weighted Product Scores – Scores each product by ratings weighted by similarity.
6. Normalization – Scores between 0–1 for direct use in your recommender.

In [34]:
# need to work on this function more - just a placeholder for now
def get_collab_scores(user_id, user_profile, full_data=full_data, top_n_sim_users=50):
    """
    Compute collaborative filtering scores for a user based on similar users
    while also considering skin profile similarity.
    
    Args:
        user_id: int, target user ID
        user_profile: dict, profile info of the user
        full_data: DataFrame, all reviews
        top_n_sim_users: int, number of similar users to consider
        
    Returns:
        dict: {product_id: score} normalized between 0-1
    """
    
    # Pivot table: users x products
    user_product_matrix = full_data.pivot_table(
        index='author_id',
        columns='product_id',
        values='weighted_rating',
        fill_value=0
    )
    
    if user_id not in user_product_matrix.index:
        return {}
    
    # Profile similarity
    profile_cols = ['skin_type','skin_tone','skin_sensitivity','eye_color','hair_color']
    
    # Map users to their profile info
    profile_df = full_data.groupby('author_id')[profile_cols].first()
    
    # Fill missing with placeholder
    profile_df = profile_df.fillna('unknown')
    
    # Encode profile features as numeric vectors
    profile_encoded = pd.get_dummies(profile_df)
    
    # Encode target user profile
    target_profile = {col: user_profile.get(col, 'unknown') for col in profile_cols}
    target_profile_df = pd.DataFrame([target_profile])
    target_profile_encoded = pd.get_dummies(target_profile_df)
    
    # Align columns
    target_profile_encoded = target_profile_encoded.reindex(columns=profile_encoded.columns, fill_value=0)
    
    # Cosine similarity between target user and all others based on profile
    profile_sim = cosine_similarity(profile_encoded, target_profile_encoded).flatten()
    
    profile_sim_df = pd.DataFrame({
        'author_id': profile_encoded.index,
        'profile_similarity': profile_sim
    })
    
    # Rating similarity
    user_vec = user_product_matrix.loc[user_id].values.reshape(1, -1)
    rating_sim = cosine_similarity(user_product_matrix, user_vec).flatten()
    
    rating_sim_df = pd.DataFrame({
        'author_id': user_product_matrix.index,
        'rating_similarity': rating_sim
    })
    
    # Combine similarities
    sim_df = profile_sim_df.merge(rating_sim_df, on='author_id')
    
    # Weighted similarity: more weight to profile (customizable)
    sim_df['similarity'] = 0.6 * sim_df['profile_similarity'] + 0.4 * sim_df['rating_similarity']
    
    # Remove the user themselves
    sim_df = sim_df[sim_df['author_id'] != user_id]
    
    # Top N similar users
    top_sim_users = sim_df.nlargest(top_n_sim_users, 'similarity')
    
    # Weighted product scores
    weighted_scores = full_data[full_data['author_id'].isin(top_sim_users['author_id'])] \
        .groupby('product_id') \
        .apply(lambda x: np.sum(
            x['weighted_rating'] * top_sim_users.set_index('author_id').loc[x['author_id'], 'similarity'].values
        )) \
        .to_dict()
    
    # Normalize scores
    if weighted_scores:
        max_score = max(weighted_scores.values())
        weighted_scores = {k: v / max_score for k, v in weighted_scores.items()}
    
    return weighted_scores

### Final Recommender System
---

In [ ]:
def recommend(user_profile, user_id=None, top_n=10):
    
    df = product_stats.copy()
    
    # ingredient score
    df['ingredient_score'] = df['ingredients'].apply(
        lambda x: ingredient_compatibility_score(x, user_profile, ingredient_risk)
    )
    
    # skin match
    if user_profile['skin_type'] in skin_match_ratio.columns:
        df['skin_match'] = df['product_id'].map(
            skin_match_ratio[user_profile['skin_type']]
        ).fillna(0)
    else:
        df['skin_match'] = 0.5
    
    # NLP
    df['sentiment'] = df['avg_sentiment'].fillna(0)
    # normalize -1 → 1 into 0 → 1
    df['sentiment'] = (df['sentiment'] + 1) / 2
    
    # collaborative filtering
    if user_id is not None:
        collab_scores = get_collab_scores(user_id, user_profile)
        df['collab'] = df['product_id'].map(collab_scores).fillna(0)
    else:
        df['collab'] = 0
    
    # normalize collab
    if df['collab'].max() > 0:
        df['collab'] = df['collab'] / df['collab'].max()

    # highlight score
    df['highlight_score'] = df['highlights'].apply(
        lambda x: highlight_score(x, user_profile)
    )
    
    # FINAL SCORE (FULL HYBRID)
    df['final_score'] = (
        0.20 * df['ingredient_score'] +
        0.20 * df['adjusted_rating'] +
        0.08 * df['popularity_score'] +
        0.05 * df['hype_score'] +
        0.20 * df['skin_match'] +
        0.07 * df['highlight_score'] +
        0.07 * df['sentiment'] +
        0.13 * df['collab']
    )
    
    return df.sort_values('final_score', ascending=False).head(top_n)